# Entrenamiento YOLO11n a 1280 para matrículas lejanas

Igual que el entrenamiento anterior de Colab, pero con `imgsz=1280` y más aumento de escala (`scale=0.9`) para detectar matrículas pequeñas. Requiere el secreto `HF_TOKEN` en Colab y GPU. Guarda en `matriculas_yolo11n_1280`.

In [ ]:
!pip install -q ultralytics huggingface_hub

In [ ]:
from google.colab import userdata
from huggingface_hub import snapshot_download

snapshot_download("YoelRuso/MatriculasVC", repo_type="dataset", local_dir="datos",
                  allow_patterns=["README.md", "dataset_limpio/**"],
                  token=userdata.get("HF_TOKEN"))

In [ ]:
from pathlib import Path
import torch
from ultralytics import YOLO

DATASET = Path("datos/dataset_limpio").resolve()
RUNS = Path("runs").resolve()
DEVICE = 0 if torch.cuda.is_available() else "cpu"
print("Dispositivo:", DEVICE)

# batch=4: a 1280 la memoria se dispara (con 640 usabas 16); si da CUDA out of memory, bájalo a 2
YOLO("yolo11n.pt").train(data=str(DATASET / "data.yaml"), epochs=60, imgsz=1280, batch=4,
                         scale=0.9, mosaic=1.0, device=DEVICE, workers=2, amp=False,
                         project=str(RUNS), name="matriculas_yolo11n_1280", exist_ok=True)

In [ ]:
m = YOLO("runs/matriculas_yolo11n_1280/weights/best.pt").val(data=str(DATASET / "data.yaml"), split="test", imgsz=1280)
print(f"Test | Precisión {m.box.mp:.3f} | Recall {m.box.mr:.3f} | mAP50 {m.box.map50:.3f} | mAP50-95 {m.box.map:.3f}")

In [ ]:
!zip -r matriculas_yolo11n_1280.zip runs/matriculas_yolo11n_1280
from google.colab import files
files.download("matriculas_yolo11n_1280.zip")